In [3]:
import numpy as np
from scipy.stats import truncnorm
import astropy.cosmology as cosmo
from astropy.cosmology import Planck18 as cosmo
import os
import pandas as pd
from constants.z_to_D import z_to_Dz

z0=1
Dz0 = z_to_Dz(z0)/1e6  # 转换为Gpc

def create_skewed_gaussian(mean, upper_err, lower_err, size=1):
    """创建偏高斯分布，根据上下误差不对称"""
    if upper_err == lower_err:
        return np.random.normal(mean, upper_err, size)
    else:
        # 使用截断正态分布来模拟偏分布
        scale = (upper_err + lower_err) / 2
        skew_factor = (upper_err - lower_err) / (upper_err + lower_err)
        samples = truncnorm.rvs(-2, 2, loc=mean + skew_factor, scale=scale, size=size)
        return np.clip(samples, mean - lower_err, mean + upper_err)

def sample_m1(N, alpha1, alpha2, mu1, mu2, sigma1=0.3, sigma2=3.0):
    """根据给定参数采样主质量"""
    lambda0, lambda1, lambda2 = 0.324, 0.634, 0.042  # λ₀=0.324, λ₁=0.634, λ₂=0.041
    comp_choice = np.random.choice([0, 1, 2], size=N, p=[lambda0, lambda1, lambda2])
    
    m1_samples = np.zeros(N)
    m_break = 20.0
    m_low, m_high = 5.0, 100.0
    
    # 1. 断裂幂律部分
    mask_bpl = (comp_choice == 0)
    n_bpl = mask_bpl.sum()
    if n_bpl > 0:
        u = np.random.uniform(0, 1, n_bpl)
        
        # 计算断裂点两侧的归一化常数
        A_low = (m_break**(-alpha1) - m_low**(-alpha1)) / alpha1
        A_high = (m_high**(-alpha2) - m_break**(-alpha2)) / alpha2
        frac_low = A_low / (A_low + A_high)
        
        low_mask = u < frac_low
        
        # 低质量段采样
        u_low = np.random.uniform(0, 1, low_mask.sum())
        m1_low = (m_low**(-alpha1) + u_low * (m_break**(-alpha1) - m_low**(-alpha1)))**(-1/alpha1)
        
        # 高质量段采样
        u_high = np.random.uniform(0, 1, (~low_mask).sum())
        m1_high = (m_break**(-alpha2) + u_high * (m_high**(-alpha2) - m_break**(-alpha2)))**(-1/alpha2)
        
        m1_samples[mask_bpl] = np.concatenate([m1_low, m1_high])
    
    # 2. 第一个高斯峰
    mask_g1 = (comp_choice == 1)
    if mask_g1.sum() > 0:
        m1_samples[mask_g1] = truncnorm.rvs(
            (m_low - mu1)/sigma1, (m_high - mu1)/sigma1, loc=mu1, scale=sigma1, size=mask_g1.sum()
        )
    
    # 3. 第二个高斯峰
    mask_g2 = (comp_choice == 2)
    if mask_g2.sum() > 0:
        m1_samples[mask_g2] = truncnorm.rvs(
            (m_low - mu2)/sigma2, (m_high - mu2)/sigma2, loc=mu2, scale=sigma2, size=mask_g2.sum()
        )
    
    return m1_samples

# def generate_redshift_distribution(N_sources, kappa, z_max=z0):
#     """生成红移数据"""
#     z_values = np.linspace(0, z_max, 500)
#     dVcdz = cosmo.differential_comoving_volume(z_values).value
#     pz = (1 + z_values)**(kappa - 1) * dVcdz
#     pz_norm = pz / np.trapz(pz, z_values)
#     z_samples = np.random.choice(z_values, size=N_sources, p=pz_norm/np.sum(pz_norm))
#     return z_samples

def generate_redshift_distribution(N_sources, kappa, z_max=z0):
    z_values = np.linspace(0, z_max, 5000)
    dVcdz = cosmo.differential_comoving_volume(z_values).value
    pz = (1 + z_values)**(kappa - 1) * dVcdz

    # 连续归一化
    pz = pz / np.trapz(pz, z_values)

    dz = z_values[1] - z_values[0]
    cdf = np.cumsum(pz) * dz
    cdf = cdf / cdf[-1]

    u = np.random.rand(N_sources)
    z_samples = np.interp(u, cdf, z_values)

    return z_samples

def generate_catalog_from_rate(catalog_id, observation_time, volume, min_sources):
    """
    根据并合率计算源数量
    
    Parameters:
    - catalog_id: 目录ID
    - observation_time: 观测时间 (年)
    - volume: 观测体积 (Gpc^3)
    - min_sources: 最小源数量
    """
    # 随机生成参数（考虑误差范围）
    alpha1 = create_skewed_gaussian(1.7, 1.2, 1.8)[0]  # 1.7^{+1.2}_{-1.8}
    alpha2 = create_skewed_gaussian(4.5, 1.6, 1.3)[0]  # 4.5^{+1.6}_{-1.3}
    beta_q = create_skewed_gaussian(1.2, 1.2, 1.0)[0]  # 1.2^{+1.2}_{-1.0}
    mu1 = create_skewed_gaussian(9.8, 0.3, 0.6)[0]     # 9.8^{+0.3}_{-0.6}
    mu2 = create_skewed_gaussian(32.7, 2.7, 6.5)[0]    # 32.7^{+2.7}_{-6.5}
    kappa = create_skewed_gaussian(3.2, 0.94, 1.00)[0] # 3.2^{+0.94}_{-1.00}
    
    # 并合率在范围内均匀分布
    merger_rate = np.random.uniform(14, 26)
    
    # 计算预期源数量 (并合率 × 共动体积 × 时间)
    expected_N = merger_rate * volume * observation_time
    
    # 使用泊松分布生成实际源数量
    N_sources = np.random.poisson(expected_N)
    
    # 确保至少有一定数量的源
    N_sources = max(N_sources, min_sources)
    
    print(f"Catalog {catalog_id:2d}: 并合率={merger_rate:.1f}, 共动体积={volume} Gpc³, 时间={observation_time} yr")
    print(f"               预期源数={expected_N:.1f}, 实际源数={N_sources}")
    print(f"               参数: α1={alpha1:.2f}, α2={alpha2:.2f}, βq={beta_q:.2f}, κ={kappa:.2f}")
    
    # ===== 红移分布 =====
    z_samples = generate_redshift_distribution(N_sources, kappa)
    
    # ===== 主质量分布 =====
    m1_samples = sample_m1(N_sources, alpha1, alpha2, mu1, mu2)
    
    # ===== 质量比分布 =====
    def sample_q(m1, beta_q, m2_low=3.0):
        q_min = m2_low / m1
        q_max = 1.0
        u = np.random.uniform(0, 1, len(m1))
        q_samples = (q_min**(beta_q+1) + u * (q_max**(beta_q+1) - q_min**(beta_q+1)))**(1/(beta_q+1))
        return q_samples
    
    q_samples = sample_q(m1_samples, beta_q)
    m2_samples = q_samples * m1_samples
    
    # ===== 其他参数（均匀分布） =====
    t_c_samples = np.random.uniform(0, observation_time, N_sources)  # 并合时间 (yr)
    psi_samples = np.random.uniform(0, np.pi, N_sources)  # 偏振角 (rad)
    
    # ===== 构建DataFrame =====
    df = pd.DataFrame({
        'z': z_samples,
        'm1_Msun': m1_samples,
        'm2_Msun': m2_samples,
        't_c_yr': t_c_samples,
        'psi_rad': psi_samples
    })
    
    return df, {
        'alpha1': alpha1, 'alpha2': alpha2, 'beta_q': beta_q,
        'mu1': mu1, 'mu2': mu2, 'kappa': kappa, 
        'merger_rate': merger_rate, 'N_sources': N_sources,
        'observation_time': observation_time, 'volume': volume,
        'expected_N': expected_N
    }

# ===== 生成100个catalog =====
N_catalogs = 1

# 创建输出目录
output_dir = '/data/chexy/BP2P2026/bbh_catalogs'
os.makedirs(output_dir, exist_ok=True)

# 存储所有catalog的参数
catalog_params = []
total_sources = 0

print("根据并合率生成100个双黑洞catalog (CSV格式):")
print("=" * 80)

for i in range(N_catalogs):
    # 可以为每个catalog设置不同的观测参数
    observation_time = 5  # 观测时间 5年
    volume = 4/3*np.pi*Dz0**3  # 体积约为 4.19 Gpc³ (半径1Gpc的球体)
    
    df, params = generate_catalog_from_rate(
        catalog_id=i+1, 
        observation_time=observation_time, 
        volume=volume,
        min_sources=20  # 确保每个catalog至少有20个源
    )
    
    # 保存为CSV文件
    filename = f'{output_dir}/catalog_{i+1:03d}.csv'
    df.to_csv(filename, index=False)
    
    catalog_params.append(params)
    total_sources += params['N_sources']

# 保存参数汇总
param_summary = pd.DataFrame(catalog_params)
param_summary.to_csv(f'{output_dir}/catalog_parameters_summary.csv', index=False)

print("=" * 80)
print(f"生成完成！共 {N_catalogs} 个catalog")
print(f"总源数量: {total_sources}")
print(f"平均每个catalog: {total_sources/N_catalogs:.1f} 个源")
print(f"最小catalog: {min([p['N_sources'] for p in catalog_params])} 个源")
print(f"最大catalog: {max([p['N_sources'] for p in catalog_params])} 个源")
print(f"输出目录: {output_dir}/")

# 显示参数统计
print(f"\n参数统计:")
print(f"α1: {param_summary['alpha1'].mean():.2f} ± {param_summary['alpha1'].std():.2f}")
print(f"α2: {param_summary['alpha2'].mean():.2f} ± {param_summary['alpha2'].std():.2f}")
print(f"βq: {param_summary['beta_q'].mean():.2f} ± {param_summary['beta_q'].std():.2f}")
print(f"μ1: {param_summary['mu1'].mean():.1f} ± {param_summary['mu1'].std():.1f}")
print(f"μ2: {param_summary['mu2'].mean():.1f} ± {param_summary['mu2'].std():.1f}")
print(f"κ: {param_summary['kappa'].mean():.2f} ± {param_summary['kappa'].std():.2f}")
print(f"并合率: {param_summary['merger_rate'].mean():.1f} ± {param_summary['merger_rate'].std():.1f} Gpc-3 yr-1")
print(f"观测时间: {param_summary['observation_time'].mean():.1f} ± {param_summary['observation_time'].std():.1f} 年")
print(f"观测体积: {param_summary['volume'].mean():.1f} ± {param_summary['volume'].std():.1f} Gpc³")

# 显示第一个catalog的前几行作为示例
print(f"\n第一个catalog的前5行示例:")
try:
    example_df = pd.read_csv(f'{output_dir}/catalog_001.csv')
    print(example_df.head())
except Exception as e:
    print(f"读取示例文件失败: {e}")

根据并合率生成100个双黑洞catalog (CSV格式):
Catalog  1: 并合率=23.2, 共动体积=162.17402347565073 Gpc³, 时间=5 yr
               预期源数=18791.9, 实际源数=18862
               参数: α1=0.07, α2=3.98, βq=1.55, κ=3.86
生成完成！共 1 个catalog
总源数量: 18862
平均每个catalog: 18862.0 个源
最小catalog: 18862 个源
最大catalog: 18862 个源
输出目录: /data/chexy/BP2P2026/bbh_catalogs/

参数统计:
α1: 0.07 ± nan
α2: 3.98 ± nan
βq: 1.55 ± nan
μ1: 9.2 ± nan
μ2: 29.0 ± nan
κ: 3.86 ± nan
并合率: 23.2 ± nan Gpc-3 yr-1
观测时间: 5.0 ± nan 年
观测体积: 162.2 ± nan Gpc³

第一个catalog的前5行示例:
          z    m1_Msun    m2_Msun    t_c_yr   psi_rad
0  0.815425  10.977131  10.369820  2.462546  1.416294
1  0.655314   8.973317   7.690328  3.781458  0.704481
2  0.696788   9.052796   7.109216  4.173529  0.693816
3  0.817472   9.201044   7.982359  4.808752  1.304901
4  0.874434   9.656546   7.430694  0.417562  2.786615
